In [ ]:
!pip install openai matplotlib koreanize-matplotlib

In [ ]:
# ============================================================
# 실습 2-4. 연구 가설 생성과 Temperature 실험
# 목표: T별 가설 다양성을 TTR로 수치화하고, 멀티턴으로 가설을 구체화한다
# 사전 설치: pip install openai matplotlib koreanize-matplotlib
# ============================================================

import os
import numpy as np
import matplotlib.pyplot as plt
import koreanize_matplotlib
plt.rcParams['axes.unicode_minus'] = False

# ── API 키 설정 ──────────────────────────────────────────────
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("Colab Secrets에서 API 키 로드 완료")
except Exception:
    pass
# os.environ["OPENAI_API_KEY"] = "sk-..."

if not os.environ.get("OPENAI_API_KEY"):
    raise ValueError(
        "API 키가 설정되지 않았습니다.\n"
        "방법 A: 왼쪽 사이드바 열쇠 아이콘 -> 'OPENAI_API_KEY' 등록\n"
        "방법 B: os.environ['OPENAI_API_KEY'] = 'sk-...' 주석 해제"
    )

from openai import OpenAI
client = OpenAI()
print(f"API 키 확인: sk-...{os.environ['OPENAI_API_KEY'][-4:]}")


# ============================================================
# [실습 변수] 여기만 수정하여 다양한 실험 가능
# ============================================================

# 가설 생성 기반 연구 배경
# 실습 2-3에서 분석한 Attention 논문의 한계에서 출발
# 본인 연구 배경으로 교체 가능
RESEARCH_BACKGROUND = """
연구 분야: 자연어처리 / 트랜스포머 아키텍처 응용
기반 논문: Attention Is All You Need (Vaswani et al., 2017)
기존 한계: 트랜스포머는 고정 길이 컨텍스트 윈도우로 인해
           매우 긴 문서(논문 전문, 법률 문서 등) 처리에 제약이 있음
보유 환경: GPU 클러스터, 한국어 과학기술 논문 코퍼스 10만 건
"""

# 비교할 Temperature 목록
# 낮을수록 보수적, 높을수록 창의적
HYP_TEMPS = [0.1, 0.7, 1.2]

# 온도별 생성할 가설 수
N_HYPOTHESES = 3

# 멀티턴 가설 구체화 대화 질문 목록
# 본인 연구 질문으로 교체 가능
MULTITURN_QUESTIONS = [
    "트랜스포머의 고정 컨텍스트 윈도우 한계를 극복하면 긴 논문 전문 처리 성능이 향상될 것이다. 이 가설을 어떻게 검증할 수 있을까?",
    "Longformer나 BigBird 같은 기존 long-context 모델과의 비교 실험은 어떻게 설계할까?",
    "한국어 과학기술 논문에 특화된 평가 지표는 무엇이 적합할까?",
]

# Temperature 시각화용 엔트로피 계산 기준 logit 값
# 다음 토큰 후보 4개에 대한 가상의 점수
LOGITS_SAMPLE = [3.0, 2.5, 1.5, 1.0]

# ============================================================
# 이하 코드는 수정 불필요
# ============================================================

# ── 공통 함수 ─────────────────────────────────────────────────
def chat(system_prompt, user_prompt,
         temperature=0.7, max_tokens=500):
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return resp.choices[0].message.content, resp.usage

def section(title):
    print(f"\n{'='*60}")
    print(f"  {title}")
    print(f"{'='*60}")

def type_token_ratio(text):
    """
    TTR(Type-Token Ratio): 어휘 다양성 측정 지표
    TTR = 고유 단어 수 / 전체 단어 수
    높을수록 다양한 어휘 사용, 낮을수록 반복 표현 많음
    주의: Temperature가 높다고 항상 TTR이 높지 않음
    -> 과도하게 높으면 비표준 단어 반복으로 TTR이 오히려 낮아질 수 있음
    """
    words = text.split()
    if not words:
        return 0.0
    return len(set(words)) / len(words)

def temperature_softmax(logits, temperature):
    """Temperature 적용 후 softmax 계산"""
    scaled = np.array(logits) / max(temperature, 1e-5)
    exp    = np.exp(scaled - np.max(scaled))
    return exp / exp.sum()


# ── 블록 1: 온도별 가설 생성 ─────────────────────────────────
section("블록 1. Temperature별 연구 가설 생성")

sys_hypothesis = """역할: 당신은 창의적인 연구 방법론 전문가입니다.
지시: 주어진 연구 배경을 바탕으로 검증 가능한 연구 가설을 생성하세요.
형식: 각 가설을 "H[번호]: [가설 내용]" 형태로 작성하세요.
조건: 구체적이고 측정 가능한 형태로 작성하세요."""

print(f"\n[연구 배경]")
print(RESEARCH_BACKGROUND)

hypothesis_results = {}
for t in HYP_TEMPS:
    ans, _ = chat(
        system_prompt=sys_hypothesis,
        user_prompt=(
            f"다음 연구 배경을 바탕으로 가설 {N_HYPOTHESES}개를 제시하세요:\n"
            f"{RESEARCH_BACKGROUND}"
        ),
        temperature=t,
        max_tokens=400,
    )
    hypothesis_results[t] = ans
    label = {
        0.1: "보수적 (기존 트랜스포머 연장선)",
        0.7: "균형적 (표준적 연구 방향)",
        1.2: "창의적 (새로운 시각)",
    }.get(t, "")
    print(f"\n[Temperature={t}]  {label}")
    print(ans)


# ── 블록 2: TTR 측정 및 시각화 ───────────────────────────────
section("블록 2. 어휘 다양성(TTR) 측정 및 엔트로피 비교")

ttrs = [type_token_ratio(hypothesis_results[t]) for t in HYP_TEMPS]
temp_labels = [f"T={t}" for t in HYP_TEMPS]
colors_bar  = ["#B5D4F4", "#9FE1CB", "#FAC775"]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 왼쪽: TTR 막대 그래프
bars = axes[0].bar(temp_labels, ttrs, color=colors_bar,
                   edgecolor="none", width=0.45)
for bar, v in zip(bars, ttrs):
    axes[0].text(bar.get_x() + bar.get_width()/2, v + 0.008,
                 f"{v:.3f}", ha='center', fontsize=11)
axes[0].set_ylabel("어휘 다양성 (TTR)")
axes[0].set_title(
    "Temperature vs 가설 어휘 다양성\n(높을수록 다양한 표현)",
    fontsize=11
)
axes[0].set_ylim(0, 1.0)
axes[0].axhline(0.7, color='gray', linewidth=0.8, linestyle='--',
                label='TTR=0.7 기준선')
axes[0].legend(fontsize=9)

# 오른쪽: 엔트로피 곡선 (실습 2-1 복습)
t_range   = np.linspace(0.1, 2.0, 60)
entropies = []
for t in t_range:
    p = temperature_softmax(LOGITS_SAMPLE, t)
    entropies.append(-np.sum(p * np.log(p + 1e-9)))

axes[1].plot(t_range, entropies, color='#378ADD', linewidth=2)
for t, c in zip(HYP_TEMPS, colors_bar):
    axes[1].axvline(t, color=c, linestyle='--', linewidth=1.8,
                    label=f"T={t}")
axes[1].set_xlabel("Temperature")
axes[1].set_ylabel("엔트로피")
axes[1].set_title(
    "전날 복습: Temperature vs 엔트로피\n(API 실험 결과와 비교)",
    fontsize=11
)
axes[1].legend(fontsize=9)

plt.suptitle("Temperature 실험 - 연구 가설 생성", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("hypothesis_temperature.png", dpi=150, bbox_inches='tight')
plt.show()

print("\nTTR 측정 결과:")
for t, ttr in zip(HYP_TEMPS, ttrs):
    print(f"  T={t}  TTR={ttr:.3f}")
print("\n주의: T가 가장 높다고 TTR이 항상 최고는 아닙니다.")
print("      과도한 온도에서는 비표준 단어 반복으로 TTR이 낮아질 수 있습니다.")
print("\n연구 단계별 Temperature 권장:")
print("  아이디어 탐색 (T=1.0~1.2): 다양한 가설 후보 생성")
print("  연구 설계   (T=0.5~0.7) : 균형 잡힌 방법론")
print("  논문 작성   (T=0.1~0.3) : 사실 기반 서술")


# ── 블록 3: 멀티턴 가설 구체화 대화 ──────────────────────────
section("블록 3. 멀티턴 가설 구체화 대화")

# 멀티턴 구조: 매 요청마다 전체 이력을 history에 담아 전달
# 장점: 이전 대화 문맥 유지
# 주의: 대화가 길수록 토큰 누적 -> 비용 증가
history = [
    {
        "role": "system",
        "content": (
            "당신은 NLP 연구 방법론 전문가입니다. "
            "연구자의 가설을 단계적으로 구체화하도록 도와주세요."
        )
    }
]

for i, question in enumerate(MULTITURN_QUESTIONS, 1):
    # 사용자 질문을 이력에 추가
    history.append({"role": "user", "content": question})

    # 전체 이력을 포함해 API 호출 -> 이전 맥락 참조 가능
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=history,
        temperature=0.5,
        max_tokens=300,
    )
    answer = resp.choices[0].message.content
    # 모델 응답도 이력에 추가 -> 다음 턴에서 참조
    history.append({"role": "assistant", "content": answer})

    print(f"\n[{i}턴] 연구자: {question}")
    print(f"       AI:    {answer[:250]}...")
    print(f"       (누적 이력: {len(history)-1}개 메시지)")

total_turns = (len(history) - 1) // 2
print(f"\n누적 대화 턴: {total_turns}턴")
print("-> 멀티턴 구조: 매 요청마다 전체 이력 전달 - 토큰 누적 주의")
print("-> 긴 대화는 중간에 요약 삽입하거나 오래된 이력을 삭제하는 전략 필요")
print("-> 다음 실습: API의 한계(내부 문서 부재, 환각)를 직접 체험합니다")
